# 1.使用预定义工具(Tavily)

In [1]:
import pydantic
from langchain_core.messages import HumanMessage
from langchain_tavily import TavilySearch
from dotenv import load_dotenv
import os

from pydantic import BaseModel

load_dotenv()

# search_tool是一个工具,可直接调用或让agent调用
search_tool = TavilySearch(
    max_results=5,
    topic='general'
)

In [2]:
search_tool.invoke("原神好玩吗？")

{'query': '原神好玩吗？',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://forum.gamer.com.tw/C.php?bsn=36730&snA=43202',
   'title': '【攻略】現在的原神還適合入坑嗎 @原神 哈啦板 - 巴哈姆特',
   'content': 'LV. 46\n\nGP 12k\n\n \n\n11 樓 三角初華 Faithcate\n\nGP1 BP-\n\n2025-07-24 07:52:02 編輯 \ue32c\n\n可以玩，但時間不多的玩家，及對高搜索率沒耐心的話，我是真的不知道要把所有內容玩完需要花多少時間，如果要上班工作的話\n\n加上有些任有解謎成份，不看攻略要解很久，但沒單機玩的話，這款可秏掉很多時間了，我是覺得現在玩家都有同時玩多款遊戲的狀況\n\n角色5.8還能抽，有護盾的角色可幫很多忙，然後開始抽6.0角，環境應該又會變了，只是新玩家是否有耐心養這些角色，培養過程沒像其他遊戲這麼簡單，要很有耐心啊\n\n1\n\n-\n\n  \n\nLV. 49\n\nGP 2k\n\n \n\n12 樓 真‧神無月 sony991419\n\nGP2 BP-\n\n2025-07-24 23:49:51\n\n只要排除了深淵，基本上什麼時間入坑也沒差吧？\n\n原神的大部份趣味都不在深淵，深淵佔整體無課福利很少部份\n\n當你能無視深淵後角色命座可不抽，武器也不一定要(除非你喜歡那武器外觀)，沒了一些原石但抽卡壓力反而下降了 [...] 我是覺得沒有因此而變得不好玩，畢竟新手時期的新鮮感濾鏡還是很重的\n\n我也有個草國時期才加入的新手朋友，那時卡池是散兵一斗，他不抽男又對雷神沒興趣\n\n一路玩到夜蘭復刻才抽第一支限\n\n但我看他還是玩得很開心，也問我很多，像是反應怎麼解好，飛在空中的機械要怎麼打好\n\n而我那時在弄論文很忙也不太有空帶，基本都只回訊息交一些東西或丟攻略文章，讓他自己慢慢玩\n\n他還比我更早打完草國主線和森林書還探索100%\n\n而且現在四星角真的很強，能玩出一些配隊和反應，還有97忍、嘉明這種傷害不錯的4星\n\n另外你提到的次數盾打很

In [7]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from pydantic import BaseModel, Field


# BaseModel是一种数据验证，解析和设置的类，用于告诉大模型输出数据的格式(模版)
# Field用于给字段添加元数据，例如描述，最大/小指等
class Reference(BaseModel):
    title: str = Field(description="The title of the web page cited in the answer.")
    url: str = Field(description="The URL of the web page cited in the answer.")

class AnswerInfo(BaseModel):
    answer: str = Field(description="The final answer to the question.")
    references: list[Reference] = Field(description="The web pages cited in the answer.")


agent = create_agent(
    model='deepseek-chat',
    tools=[search_tool],
    system_prompt="你是一个 helpful的助手，可以使用预定义的工具来帮助用户。",
    response_format=AnswerInfo,
)

response = agent.invoke(
    {"messages" : [HumanMessage(content="蔚蓝档案好玩吗？")]}
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

蔚蓝档案好玩吗？
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_00_6wu0sXTKeuqPdKPA0Ciz8895)
 Call ID: call_00_6wu0sXTKeuqPdKPA0Ciz8895
  Args:
    query: 蔚蓝档案 游戏评价 好玩吗 玩家口碑
    search_depth: advanced
  tavily_search (call_01_chIQVSwyvOlYWXxr5yaQ5245)
 Call ID: call_01_chIQVSwyvOlYWXxr5yaQ5245
  Args:
    query: Blue Archive review gameplay pros cons
    search_depth: advanced
================================= Tool Message =================================
Name: tavily_search

{"query": "蔚蓝档案 游戏评价 好玩吗 玩家口碑", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.taptap.cn/app/316964/review", "title": "蔚蓝档案- 游戏评价", "content": "蔚蓝档案 · 1.总体评价：我个人认为这是一个非常优秀的游戏 · 2.个人经历：我是开服第二天入的坑，在此之前没有接触过日服和国际服，属于完完全全的萌新。 · 3.剧情：ba的剧情", "score": 0.8820324, "raw_content": null, "id": "b9f387-00"}, {"url": "https://www.gamersky.com